In [0]:
dbutils.widgets.text("login", "")
dbutils.widgets.text("storage_account", "")
dbutils.widgets.text("secret_scope", "")
dbutils.widgets.text("container", "")

In [0]:
login = dbutils.widgets.get("login")
storage_account = dbutils.widgets.get("storage_account")
container = dbutils.widgets.get("container")
secret_scope = dbutils.widgets.get("secret_scope")
dbutils.secrets.list(scope=secret_scope)

In [0]:
client_id = dbutils.secrets.get(scope=secret_scope, key="sp-databricks-adls-appid")
client_secret = dbutils.secrets.get(scope=secret_scope, key="sp-databricks-adls-appkey")
tenant_id = dbutils.secrets.get(scope=secret_scope, key="tenant-id")

In [0]:
configs = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint":
        f"https://login.microsoftonline.com/{tenant_id}/oauth2/token",
}

In [0]:
mount_point = f"/mnt/{login}"
source = f"abfss://{container}@{storage_account}.dfs.core.windows.net/"

if any(m.mountPoint == mount_point for m in dbutils.fs.mounts()):
    dbutils.fs.unmount(mount_point)

dbutils.fs.mount(source=source, mount_point=mount_point, extra_configs=configs)
print(f"Zamontowano {source} pod {mount_point}")

# ('Method public com.databricks.backend.daemon.dbutils.DBUtilsCore$Result com.databricks.backend.daemon.dbutils.DBUtilsCore.mounts() is not whitelisted on class class com.databricks.backend.daemon.dbutils.DBUtilsCore',)

This is not working because mounts are a legacy mechanism that bypasses Unity Catalog and would expose the storage to all cluster users via the SPN identity, so UC Shared mode disallows them.